# FinanceBench RAG Q&A System + Evaluation



In [ ]:
!pip install -q pymupdf sentence-transformers faiss-cpu groq gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 6.4 MB/s eta 0:00:00


In [ ]:
import os, re, json, getpass
import numpy as np
import pandas as pd
import pymupdf
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 80)

## Step 2: Dataset load (mee CSV)

In [ ]:
from google.colab import files
uploaded = files.upload()   # financebench_data.csv ni select cheyyandi

Saving financebench_data.csv to financebench_data.csv


In [ ]:
df = pd.read_csv("financebench_data.csv")

# "80;81" -> [80, 81]
df["gold_pages"] = df["evidence_page_num"].astype(str).apply(
    lambda x: [int(p) for p in x.split(";")]
)

print(df.shape)
df[["company", "doc_name", "question_type", "gold_pages"]]

(8, 13)


,company,doc_name,question_type,gold_pages
0,American Express,AMERICANEXPRESS_2022_10K,novel-generated,[44]
1,American Water Works,AMERICANWATERWORKS_2020_10K,metrics-generated,[85]
2,American Water Works,AMERICANWATERWORKS_2021_10K,metrics-generated,"[85, 87]"
3,American Water Works,AMERICANWATERWORKS_2022_10K,domain-relevant,"[80, 81]"
4,American Water Works,AMERICANWATERWORKS_2022_10K,domain-relevant,"[80, 81]"
5,Best Buy,BESTBUY_2017_10K,metrics-generated,[55]
6,Best Buy,BESTBUY_2019_10K,metrics-generated,[51]
7,Best Buy,BESTBUY_2023_10K,domain-relevant,[39]


In [ ]:
# Same question rendu sarlu unte (different answer wording tho), okate unique question ga teesukundam
print("Rows:", len(df), "| Unique questions:", df["question"].nunique())
df["question"].value_counts().head(3)

Rows: 8 | Unique questions: 7


,count
question,
"Does American Water Works have positive working capital based on FY2022 data? If working capital is not a useful or relevant metric for this company, then please state that and explain why.",2
How much (in USD billions) did American Water Works pay out in cash dividends for FY2020? Compute or extract the answer by primarily using the details outlined in the statement of cash flows.,1
Was American Express able to retain card members during 2022?,1


## Step 3: 10-K PDFs download
.

In [ ]:
!git clone --depth 1 https://github.com/patronus-ai/financebench.git

Cloning into 'financebench'...
remote: Enumerating objects: 391, done.
remote: Counting objects: 100% (391/391), done.
remote: Compressing objects: 100% (371/371), done.
remote: Total 391 (delta 18), reused 389 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (391/391), 551.42 MiB | 31.45 MiB/s, done.
Resolving deltas: 100% (18/18), done.
Updating files: 100% (390/390), done.


In [ ]:
PDF_DIR = Path("financebench/pdfs")
needed_docs = sorted(df["doc_name"].unique())

for d in needed_docs:
    print(d, "->", (PDF_DIR / f"{d}.pdf").exists())

AMERICANEXPRESS_2022_10K -> True
AMERICANWATERWORKS_2020_10K -> True
AMERICANWATERWORKS_2021_10K -> True
AMERICANWATERWORKS_2022_10K -> True
BESTBUY_2017_10K -> True
BESTBUY_2019_10K -> True
BESTBUY_2023_10K -> True


## Step 4: PDF nundi page-wise text

In [ ]:
pages_by_doc = {}   # {doc_name: [page0_text, page1_text, ...]}

for d in needed_docs:
    with pymupdf.open(PDF_DIR / f"{d}.pdf") as pdf:
        pages_by_doc[d] = [page.get_text() for page in pdf]
    print(d, "->", len(pages_by_doc[d]), "pages")

AMERICANEXPRESS_2022_10K -> 260 pages
AMERICANWATERWORKS_2020_10K -> 152 pages
AMERICANWATERWORKS_2021_10K -> 181 pages
AMERICANWATERWORKS_2022_10K -> 306 pages
BESTBUY_2017_10K -> 109 pages
BESTBUY_2019_10K -> 107 pages
BESTBUY_2023_10K -> 75 pages


In [ ]:

def numbers_in(text):
    return set(re.findall(r"\d{1,3}(?:,\d{3})+|\d{3,}", text))

for _, r in df.iterrows():
    nums = numbers_in(r["evidence_text"])
    page_text = " ".join(pages_by_doc[r["doc_name"]][p] for p in r["gold_pages"])
    found = sum(n in page_text for n in nums) / len(nums) if nums else None
    print(r["doc_name"], r["gold_pages"], "->", found)

AMERICANEXPRESS_2022_10K [44] -> 1.0
AMERICANWATERWORKS_2020_10K [85] -> 1.0
AMERICANWATERWORKS_2021_10K [85, 87] -> 1.0
AMERICANWATERWORKS_2022_10K [80, 81] -> 1.0
AMERICANWATERWORKS_2022_10K [80, 81] -> 1.0
BESTBUY_2017_10K [55] -> 1.0
BESTBUY_2019_10K [51] -> 1.0
BESTBUY_2023_10K [39] -> 1.0


## Step 5: Chunking + Embeddings + FAISS


In [ ]:
def split_text(text, chunk_size=1000, overlap=150):
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    if len(text) <= chunk_size:
        return [text]
    out, start = [], 0
    while start < len(text):
        out.append(text[start:start + chunk_size])
        start += chunk_size - overlap
    return out


def make_chunks(chunk_size=1000, overlap=150):
    company_of = df.drop_duplicates("doc_name").set_index("doc_name")["company"].to_dict()
    chunks = []
    for doc, pages in pages_by_doc.items():
        for page_no, page_text in enumerate(pages):
            for j, piece in enumerate(split_text(page_text, chunk_size, overlap)):
                chunks.append({
                    "chunk_id": f"{doc}_p{page_no}_c{j}",
                    "doc_name": doc,
                    "company": company_of.get(doc),
                    "page": page_no,
                    "text": piece,
                })
    return chunks

chunks = make_chunks(1000, 150)
print("Total chunks:", len(chunks))
chunks[200]

Total chunks: 5407


{'chunk_id': 'AMERICANEXPRESS_2022_10K_p34_c3',
 'doc_name': 'AMERICANEXPRESS_2022_10K',
 'company': 'American Express',
 'page': 34,
 'text': ' processes and controls related to our sales practices and business conduct generally, take disciplinary and remedial actions where appropriate, and provide information regarding our reviews to our regulators, including the Federal Reserve. We also face an increased risk of litigation and governmental and regulatory scrutiny as a result of the effects of the pandemic on market and economic conditions, such as a renewed focus on fair lending laws, and actions governmental authorities take in response to those conditions, and in connection with our ESG-related disclosures and initiatives. Legal proceedings regarding provisions in our merchant contracts, including non-discrimination and honor-all-cards provisions, could have a material adverse effect on our business and result in additional litigation and/or arbitrations, changes to our merchant a

In [ ]:
embed_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def embed_docs(texts):
    return embed_model.encode(texts, batch_size=64, normalize_embeddings=True,
                              show_progress_bar=True).astype("float32")

def embed_query(q):
    return embed_model.encode([QUERY_PREFIX + q], normalize_embeddings=True).astype("float32")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def build_index(chunks):
    # Okka global index + prathi document ki separate index
    vecs = embed_docs([c["text"] for c in chunks])
    global_index = faiss.IndexFlatIP(vecs.shape[1])
    global_index.add(vecs)

    doc_index = {}
    for doc in sorted({c["doc_name"] for c in chunks}):
        ids = [i for i, c in enumerate(chunks) if c["doc_name"] == doc]
        idx = faiss.IndexFlatIP(vecs.shape[1])
        idx.add(vecs[ids])
        doc_index[doc] = (idx, ids)
    return global_index, doc_index

global_index, doc_index = build_index(chunks)
print("Index ready:", global_index.ntotal, "vectors")

Batches:   0%|          | 0/85 [00:00<?, ?it/s]

Index ready: 5407 vectors


## Step 6: Retriever

In [14]:
def retrieve(query, k=5, doc=None):
    # doc ichhite aa document lo matrame vethukutundi
    q = embed_query(query)
    if doc is None:
        scores, ids = global_index.search(q, k)
        ids = ids[0]
    else:
        idx, id_map = doc_index[doc]
        scores, local = idx.search(q, min(k, len(id_map)))
        ids = [id_map[i] for i in local[0]]
    return [dict(chunks[i], score=float(s)) for i, s in zip(ids, scores[0])]


# Quick test
r = df.iloc[6]
print("Q:", r["question"])
for hit in retrieve(r["question"], k=3, doc=r["doc_name"]):
    print(hit["page"], round(hit["score"], 3), hit["text"][:120])
print("Gold page:", r["gold_pages"])

Q: What is the year end FY2019 total amount of inventories for Best Buy? Answer in USD millions. Base your judgments on the information provided primarily in the balance sheet.
31 0.763 Table of Contents (1) Comparable online sales are included in the comparable sales calculation. Comparable sales also ex
25 0.741  212 216 283 Total 1,238 1,514 1,581 1,632 1,732 Retail square footage (in thousands) Domestic (9) 39,500 40,360 41,039 
49 0.741 Table of Contents REPORT OF INDEPENDENT REGISTERED PUBLIC ACCOUNTING FIRM To the Board of Directors and Shareholders of 
Gold page: [51]


## Step 7: Retrieval evaluation


In [15]:
def eval_retrieval(k_list=(1, 3, 5), use_doc_filter=True):
    uq = df.drop_duplicates("question")
    max_k = max(k_list)
    hits = {k: [] for k in k_list}
    rr = []

    for _, r in uq.iterrows():
        res = retrieve(r["question"], k=max_k, doc=r["doc_name"] if use_doc_filter else None)
        gold = set(r["gold_pages"])
        good = [(h["doc_name"] == r["doc_name"]) and (h["page"] in gold) for h in res]

        for k in k_list:
            hits[k].append(any(good[:k]))
        rr.append(1 / (good.index(True) + 1) if True in good else 0)

    out = {f"Hit@{k}": round(float(np.mean(v)), 3) for k, v in hits.items()}
    out["MRR"] = round(float(np.mean(rr)), 3)
    return out

print("Doc filter tho  :", eval_retrieval(use_doc_filter=True))
print("Doc filter lekunda:", eval_retrieval(use_doc_filter=False))

Doc filter tho  : {'Hit@1': 0.429, 'Hit@3': 0.429, 'Hit@5': 0.571, 'MRR': 0.464}
Doc filter lekunda: {'Hit@1': 0.0, 'Hit@3': 0.429, 'Hit@5': 0.429, 'MRR': 0.19}


## Step 8: LLM (Groq,)

In [16]:
from groq import Groq
import getpass

try:
    from google.colab import userdata
    api_key = userdata.get("GROQ_API_KEY")
except Exception:
    api_key = getpass.getpass("Groq API key: ")

client = Groq(api_key=api_key)

# Mee account lo ippudu unna models ni Groq nundi adugutunnam
available = [m.id for m in client.models.list().data]
print(available)

['qwen/qwen3.8-27b', 'meta-llama/llama-prompt-guard-2-22m', 'allam-2-7b', 'openai/gpt-oss-20b', 'meta-llama/llama-prompt-guard-2-86m', 'openai/gpt-oss-safeguard-20b', 'canopylabs/orpheus-arabic-saudi', 'whisper-large-v3', 'canopylabs/orpheus-v1-english', 'openai/gpt-oss-120b', 'whisper-large-v3-turbo']


In [17]:
import re

# Mee account lo unna chat models (list nundi)
candidates = ["openai/gpt-oss-120b", "openai/gpt-oss-20b", "qwen/qwen3.8-27b"]
candidates = [m for m in candidates if m in available]

MODEL = None
for m in candidates:
    try:
        r = client.chat.completions.create(
            model=m,
            messages=[{"role": "user", "content": "Say hi in one word."}],
            temperature=0,
            max_tokens=1000,
        )
        print("OK     :", m, "->", r.choices[0].message.content)
        MODEL = m
        break
    except Exception as e:
        print("FAILED :", m, "->", str(e)[:300])

print("\nUsing model:", MODEL)


def llm(prompt, max_tokens=500):
    # Reasoning models aalochinchadaniki kuda tokens vadatayi, anduke extra headroom
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=max_tokens + 1000,
    )
    text = resp.choices[0].message.content or ""
    return re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()

OK     : openai/gpt-oss-120b -> Hello

Using model: openai/gpt-oss-120b


## Step 9: RAG answer function

In [18]:
def build_prompt(question, hits):
    context = "\n\n".join(
        f"[{h['doc_name']} | page {h['page']}]\n{h['text']}" for h in hits
    )
    return f"""You are a financial analyst. Answer the question using ONLY the context from the 10-K filing below.
If the answer is not in the context, say you could not find it.
For calculations, show the numbers briefly. Put the final answer on the last line.

Context:
{context}

Question: {question}
Answer:"""


def rag_answer(question, k=5, doc=None):
    hits = retrieve(question, k=k, doc=doc)
    answer = llm(build_prompt(question, hits))
    sources = sorted({(h["doc_name"], h["page"]) for h in hits})
    return answer, sources, hits


ans, src, _ = rag_answer(df.iloc[6]["question"], doc=df.iloc[6]["doc_name"])
print(ans)
print("Sources:", src)
print("Gold answer:", df.iloc[6]["answer"])

I could not find the FY2019 total amount of inventories for Best Buy in the provided excerpts.
Sources: [('BESTBUY_2019_10K', 24), ('BESTBUY_2019_10K', 25), ('BESTBUY_2019_10K', 31), ('BESTBUY_2019_10K', 49), ('BESTBUY_2019_10K', 90)]
Gold answer: $5409.00


## Step 10

In [20]:
uq = df.drop_duplicates("question").reset_index(drop=True)
rows = []

for i, r in uq.iterrows():
    ans, src, hits = rag_answer(r["question"], k=5, doc=r["doc_name"])
    rows.append({
        "id": r["financebench_id"],
        "question": r["question"],
        "gold_answer": r["answer"],
        "pred_answer": ans,
        "retrieved_pages": sorted({h["page"] for h in hits}),
        "gold_pages": r["gold_pages"],
    })
    print(f"Done {i+1}/{len(uq)}")
    time.sleep(12)   # free limit ki gap

results = pd.DataFrame(rows)
results[["question", "gold_answer", "pred_answer"]]

Done 1/7
Done 2/7
Done 3/7
Done 4/7
Done 5/7
Done 6/7
Done 7/7


,question,gold_answer,pred_answer
0,Was American Express able to retain card members during 2022?,Yes,I could not find it.
1,How much (in USD billions) did American Water Works pay out in cash dividend...,$0.40,I could not find the amount of cash dividends paid in FY 2020 in the provide...
2,Basing your judgments off of the cash flow statement and the income statemen...,$1832.00,- Operating income (FY 2021) from the Consolidated Statements of Operations:...
3,Does American Water Works have positive working capital based on FY2022 data...,Yes. American Water Works had postivie working capital of $124Mn by FY 2022.,I could not find the current‑asset figures needed to calculate working capit...
4,"In agreement with the information outlined in the income statement, what is ...",2.8%,**Calculation**\n\n| Fiscal Year | Net earnings attributable to shareholders...
5,What is the year end FY2019 total amount of inventories for Best Buy? Answer...,$5409.00,I could not find the FY 2019 total amount of inventories for Best Buy in the...
6,Are Best Buy's gross margins historically consistent (not fluctuating more t...,"Yes, the margins have been consistent, there has been a minor decline of 1.1...",I could not find the historical gross‑margin percentages in the provided exc...


## Step 11: Answer evaluation


In [21]:
def to_numbers(text):
    vals = []
    for m in re.findall(r"-?\d[\d,]*\.?\d*", str(text)):
        try:
            vals.append(float(m.replace(",", "")))
        except ValueError:
            pass
    return vals


def numeric_match(gold, pred, tol=0.01):
    g, p = to_numbers(gold), to_numbers(pred)
    if not g or not p:
        return None   # text answer (Yes/No laanti vi)
    return any(abs(p_i - g[0]) <= tol * max(abs(g[0]), 1e-9) for p_i in p)


def judge(question, gold, pred):
    prompt = f"""You are grading an answer to a financial question.

Question: {question}
Reference answer: {gold}
Model answer: {pred}

Does the model answer reach the same conclusion/number as the reference answer?
Minor wording or unit formatting differences are fine (e.g. $0.40 billion = $400 million).
Reply with exactly one word: CORRECT or INCORRECT."""
    out = llm(prompt, max_tokens=5).strip().upper()
    return out.startswith("CORRECT")

In [22]:
results["numeric_match"] = results.apply(
    lambda r: numeric_match(r["gold_answer"], r["pred_answer"]), axis=1)

judged = []
for _, r in results.iterrows():
    judged.append(judge(r["question"], r["gold_answer"], r["pred_answer"]))
    time.sleep(1)
results["judge_correct"] = judged

results["page_hit"] = results.apply(
    lambda r: any(p in r["gold_pages"] for p in r["retrieved_pages"]), axis=1)

print("LLM-judge accuracy :", round(results["judge_correct"].mean(), 3))
print("Page hit rate (k=5):", round(results["page_hit"].mean(), 3))
results[["question", "gold_answer", "judge_correct", "page_hit"]]

LLM-judge accuracy : 0.286
Page hit rate (k=5): 0.571


,question,gold_answer,judge_correct,page_hit
0,Was American Express able to retain card members during 2022?,Yes,False,False
1,How much (in USD billions) did American Water Works pay out in cash dividend...,$0.40,False,True
2,Basing your judgments off of the cash flow statement and the income statemen...,$1832.00,True,True
3,Does American Water Works have positive working capital based on FY2022 data...,Yes. American Water Works had postivie working capital of $124Mn by FY 2022.,False,True
4,"In agreement with the information outlined in the income statement, what is ...",2.8%,True,True
5,What is the year end FY2019 total amount of inventories for Best Buy? Answer...,$5409.00,False,False
6,Are Best Buy's gross margins historically consistent (not fluctuating more t...,"Yes, the margins have been consistent, there has been a minor decline of 1.1...",False,False


In [23]:
# Wrong answers vishleshana: retrieval problema, leda LLM problema?
wrong = results[~results["judge_correct"]]
for _, r in wrong.iterrows():
    print("Q   :", r["question"][:120])
    print("Gold:", r["gold_answer"])
    print("Pred:", r["pred_answer"][:200])
    print("Retrieval correct unda?", r["page_hit"])
    print("-" * 70)

Q   : Was American Express able to retain card members during 2022?
Gold: Yes
Pred: I could not find it.
Retrieval correct unda? False
----------------------------------------------------------------------
Q   : How much (in USD billions) did American Water Works pay out in cash dividends for FY2020? Compute or extract the answer 
Gold: $0.40
Pred: I could not find the amount of cash dividends paid in FY 2020 in the provided excerpts.
Retrieval correct unda? True
----------------------------------------------------------------------
Q   : Does American Water Works have positive working capital based on FY2022 data? If working capital is not a useful or rele
Gold: Yes. American Water Works had postivie working capital of $124Mn by FY 2022.
Pred: I could not find the current‑asset figures needed to calculate working capital for FY 2022 in the provided excerpts. The 10‑K excerpt lists the current‑liability items (short‑term debt $1,175 million,
Retrieval correct unda? True
---------------

## Step 12: Experiments

In [24]:
exp_rows = []

for size, ov in [(500, 75), (1000, 150), (1500, 200)]:
    chunks = make_chunks(size, ov)
    global_index, doc_index = build_index(chunks)
    m = eval_retrieval(k_list=(1, 3, 5, 10))
    exp_rows.append({"chunk_size": size, "overlap": ov, "n_chunks": len(chunks), **m})

experiments = pd.DataFrame(exp_rows)
experiments

Batches:   0%|          | 0/161 [00:00<?, ?it/s]

Batches:   0%|          | 0/85 [00:00<?, ?it/s]

Batches:   0%|          | 0/59 [00:00<?, ?it/s]

,chunk_size,overlap,n_chunks,Hit@1,Hit@3,Hit@5,Hit@10,MRR
0,500,75,10245,0.429,0.429,0.571,0.571,0.464
1,1000,150,5407,0.429,0.429,0.571,0.571,0.464
2,1500,200,3728,0.429,0.429,0.571,0.714,0.475


In [25]:

best = experiments.sort_values("MRR", ascending=False).iloc[0]
print("Best chunk size:", int(best["chunk_size"]))

chunks = make_chunks(int(best["chunk_size"]), int(best["overlap"]))
global_index, doc_index = build_index(chunks)

Best chunk size: 1500


Batches:   0%|          | 0/59 [00:00<?, ?it/s]

## Step 13: Demo UI (Gradio)

In [26]:
import gradio as gr

doc_choices = ["All documents"] + sorted(pages_by_doc.keys())

def ask(question, doc_choice):
    doc = None if doc_choice == "All documents" else doc_choice
    ans, src, hits = rag_answer(question, k=5, doc=doc)
    src_text = "\n".join(f"{d}  (page {p})" for d, p in src)
    return ans, src_text

demo = gr.Interface(
    fn=ask,
    inputs=[gr.Textbox(label="Question", lines=2), gr.Dropdown(doc_choices, value="All documents", label="Document")],
    outputs=[gr.Textbox(label="Answer"), gr.Textbox(label="Sources")],
    title="Financial 10-K Q&A (RAG)",
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://aba41113b330661d1a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
